In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from eht_inspection.alist import *


In [ ]:
avg_scan_M87_stage3 = load_alist(multi_stage=True)
avg_2s_M87_stage3 = load_alist(avg_time=2, multi_stage=True)

avg_scan_M87_stage5 = load_alist(stage=5, multi_stage=True)
avg_2s_M87_stage5 = load_alist(stage=5, avg_time=2, multi_stage=True)

### 1. SNR vs. time across stages

In [ ]:
baseline_pol_filter = {
    "baseline": ("==", "LN"),
    "polarization": ("==", "LL"),
}

fig = plot_snr_across_stages(
    {
        "3.+adhoc": avg_scan_M87_stage3,
        "5.+close": avg_scan_M87_stage5,
    },
    filters=baseline_pol_filter,
    colors=["orange", "green"],
)

### 2. Plot coherence histograms and find outliers

In [ ]:
fig, C_2s = plot_coherence_hist(
    avg_scan_M87_stage3,
    avg_2s_M87_stage3,
    threshold=0.7,
    label="M87",
)

plt.show()

In [ ]:
fig, outliers_m87 = plot_coherence_diagnostics(
    avg_scan_M87_stage3,
    C=C_2s,
    threshold=0.7,
    snr_min=7,
    source_label="M87",
)

plt.show()

### 3. Time series of multiplr quantities for specified stations and polarizations / polarization difference

For stations other than ALMA, when using only circular polarizations, the plot will exclude baseline connecting to ALMA

In [ ]:
figs = plot_quantity_vs_scan_by_station( # type: ignore
    avg_scan_M87_stage3, # pyright: ignore[reportUndefinedVariable]
    stations=["L", "A"],
    quantities="resid_phas",
    pols={
        "default": ["RR", "LL"],
        "A": ["XR", "XL", "YR", "YL"],
    },
    filters={
        "snr": (">=", 7),
    },
    title_prefix="M87 stage 3 residual phase"
)

In [ ]:
figs = plot_quantity_vs_scan_by_station(
    avg_scan_M87_stage3,
    stations=["N"],
    quantities="delay_rate",
    pols=["RR", "LL"],
    filters={"snr": (">=", 7)},
    title_prefix="M87 stage 3 delay rate"
)

In [ ]:
figs = plot_quantity_vs_scan_by_station(
    avg_scan_M87_stage3,
    stations=["N"],
    quantities="mbd",
    pols=["RR", "LL"],
    filters={"snr": (">=", 7)},
    title_prefix="M87 stage 3 MBD"
)

In [ ]:
figs = plot_quantity_vs_scan_by_station(
    avg_scan_M87_stage3,
    stations=["X"],
    quantities="mbd - sbd",
    pols=["RR", "LL"],
    filters={"snr": (">=", 7)},
    title_prefix="M87 stage 3 MBD - SBD"
)

In [ ]:
figs = plot_quantity_vs_scan_by_station(
    avg_scan_M87_stage3,
    stations=["L", "N", "A"],
    quantities=lambda d: d["mbdelay"] - d["sbdelay"],
    pols={
    "default": ["RR", "LL"],
    "A": ["XR", "YL"],
    },
    filters={"snr": (">=", 7)},
    ylabel="mbd-sbd",
    title_prefix="M87 stage 3 MBD - SBD"
)

Handle ALMA separately when calculating pol diff due to the mixed polarization.

In [ ]:
mbd_diff = add_pol_difference(
    avg_scan_M87_stage5,
    quantity="mbdelay",
    pol1="RR",
    pol2="RL",
    new_col="mbdelay_RR_minus_RL",
)

In [ ]:
figs = plot_quantity_vs_scan_by_station(
    mbd_diff,
    stations=["L", "N"],
    quantities="mbdelay_RR_minus_RL",
    pols=["RR-RL"],
    title_prefix="M87 stage 5 MBD RR - RL"
)

In [ ]:
diff = add_rl_double_difference(
    avg_scan_M87_stage5,
    quantity="mbdelay",
    new_col="mbdelay_RR_RL_minus_LR_LL",
)

figs = plot_quantity_vs_scan_by_station(
    diff,
    stations=["L"],
    quantities="mbdelay_RR_RL_minus_LR_LL",
    pols=["R-L"],
    title_prefix="M87 stage 5 MBD (RR - RL) - (LR - LL)"
)

### 4. Return outliers based on SBD-MBD, delay rate, or R-L threshold

In [ ]:
outliers = summarize_delay_outliers(
    avg_scan_M87_stage5,
    delay_col="mbdelay",
    sbd_col="sbdelay",
    rate_col="delay_rate",
    snr_col="snr",
    sbd_mbd_frac_threshold=0.2,
    rate_threshold=2.0,
    rl_dd_frac_threshold=0.2,
    snr_min=7,
    include_rl_double_difference=True,
)

In [ ]:
outliers["sbd_mbd"].head(5)

In [ ]:
outliers["delay_rate"].head(5)

In [ ]:
outliers["rl_double_difference"].head(5)

In [ ]:
outliers["all_flags"].head(5)